# v35 学習 — v28 GBM + **m34モード構造特徴**(seed dump由来18列) → LGBM×3+CatBoost×3+XGB×3
- 入力: `v35/artifacts_v35/data/train.parquet`(make_v35_parquet.py = v28 parquet + m34_列追記。PF再実行なし)
- m34_列 = 576本per-seed軌跡のコンセンサス/分位エンベロープ(行レベル) + 挑戦者モードの位置・質量・
  族別支持・GR採点・物理(井戸レベル)。**参照点はコンセンサス=モデル非依存**(blend基準は v24型なので不可)。
- origin toggle に m34 追加(切って ablation 可)。GroupKFold(5) by well。判定 = OOF apply_pp overall。
- 末尾セル: NN(combo) ブレンド + **旧v28 GBM×NN を同一行で再計算した公平基準**との直接比較。


In [2]:
# ===== 設定 + parquet ロード(pyarrow 列チャンク→事前確保 float32 = コピー1つ。RAM節約版) =====
import json, joblib, warnings, gc
from collections import Counter
import numpy as np, pandas as pd
from pathlib import Path
import pyarrow.parquet as pq
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error
warnings.filterwarnings("ignore")

ART       = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v35\artifacts_v35")
PARQUET = ART / "data" / "train.parquet"
MODELS  = ART / "models"; MODELS.mkdir(parents=True, exist_ok=True)
N_SPLITS = 5
ROW_FRAC = 1.0            # ★本家v28と同条件(frac減=精度低下が実証済)
pp_params = {"alpha": 1.0, "tau": 85, "w_pf": 0.09}

USE_ORIGIN = {"m34": True, "base": True, "pf": True, "beam": True, "self": True, "nbr": True, "zgrad": True}
PREFIXES = [("m34", "m34_"), ("self", "pf_self"), ("nbr", "pf_nbr"), ("beam", "beam_"), ("pf", "pf_")]
META = {"well", "id", "target"}
def origin_of(c):
    if c == "z_grad": return "zgrad"
    for o, pre in PREFIXES:
        if c.startswith(pre): return o
    return "base"

schema = pq.read_schema(PARQUET)
all_feats = [c for c in schema.names if c not in META]
features = [c for c in all_feats if USE_ORIGIN.get(origin_of(c), True)]
print("origin別 全特徴数:", dict(Counter(origin_of(c) for c in all_feats)))
print("USE_ORIGIN:", USE_ORIGIN, "→ 採用", len(features), "列")
ART.joinpath("features.json").write_text(json.dumps(features))
ART.joinpath("pp_params.json").write_text(json.dumps(pp_params))

nrows = pq.ParquetFile(PARQUET).metadata.num_rows
X = np.empty((nrows, len(features)), np.float32)
CHUNK = 24
for j0 in range(0, len(features), CHUNK):
    sub = features[j0:j0 + CHUNK]
    tb = pq.read_table(PARQUET, columns=sub)
    for jj, c in enumerate(sub):
        X[:, j0 + jj] = tb.column(c).to_numpy(zero_copy_only=False).astype(np.float32, copy=False)
    del tb
    if (j0 // CHUNK) % 5 == 4: gc.collect()
small = pq.read_table(PARQUET, columns=["well", "target", "last_known_tvt", "md_since", "pf_ancc"])
g  = small.column("well").to_pandas().to_numpy()
y  = small.column("target").to_numpy(zero_copy_only=False).astype(np.float32)
lk = small.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
mds = small.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
pf_abs = small.column("pf_ancc").to_numpy(zero_copy_only=False).astype(np.float64)
del small; gc.collect()
if ROW_FRAC < 0.999:
    step = max(1, int(round(1.0 / ROW_FRAC)))
    X = X[::step].copy(); y = y[::step]; g = g[::step]
    lk = lk[::step]; mds = mds[::step]; pf_abs = pf_abs[::step]
print("rows", len(X), "| features", len(features), "| wells", pd.unique(g).size,
      f"| X {X.nbytes/1e9:.1f} GB")


origin別 全特徴数: {'base': 307, 'beam': 40, 'pf': 170, 'self': 18, 'nbr': 18, 'zgrad': 1, 'm34': 21}
USE_ORIGIN: {'m34': True, 'base': True, 'pf': True, 'beam': True, 'self': True, 'nbr': True, 'zgrad': True} → 採用 575 列
rows 3783989 | features 575 | wells 773 | X 8.7 GB


In [3]:
# ===== モデル: LightGBM×3 + CatBoost×3 + XGBoost×3(v20系, GPU) =====
lgb_params = [
    dict(boosting_type="gbdt", num_leaves=255, min_child_samples=15, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.8, reg_lambda=3.0, reg_alpha=0.05, objective="regression", verbose=-1,
         n_jobs=-1, device_type="gpu", gpu_use_dp=False, max_bin=255, learning_rate=0.030, n_estimators=5000, seed=123),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=0, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=29, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
]
cb_params = [
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.020, random_seed=7),
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.030, random_seed=123),
    dict(iterations=8000, depth=7, l2_leaf_reg=3.0, min_data_in_leaf=15, border_count=192, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.010, random_seed=999),
]
xgb_params = [
    dict(n_estimators=10000, learning_rate=0.03, max_depth=6, min_child_weight=10, subsample=0.85, colsample_bytree=0.85,
         reg_alpha=0.1, reg_lambda=5.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=0, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=10000, learning_rate=0.02, max_depth=8, min_child_weight=8, subsample=0.80, colsample_bytree=0.80,
         reg_alpha=0.05, reg_lambda=4.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=1, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=12000, learning_rate=0.02, max_depth=5, min_child_weight=25, subsample=0.75, colsample_bytree=0.75,
         reg_alpha=0.5, reg_lambda=12.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=2, n_jobs=-1, early_stopping_rounds=250),
]
VARIANTS = ([(f"lightgbm-{i+1}", "lgb", p) for i, p in enumerate(lgb_params)] +
            [(f"catboost-{i+1}", "cb", p) for i, p in enumerate(cb_params)] +
            [(f"xgboost-{i+1}", "xgb", p) for i, p in enumerate(xgb_params)])
print("variants:", [v[0] for v in VARIANTS])

variants: ['lightgbm-1', 'lightgbm-2', 'lightgbm-3', 'catboost-1', 'catboost-2', 'catboost-3', 'xgboost-1', 'xgboost-2', 'xgboost-3']


In [ ]:
# ===== GroupKFold(5) 学習 → OOF + fold pkl保存(再開可) =====
def fit_one(kind, params, Xtr, ytr, Xva, yva):
    if kind == "lgb":
        m = LGBMRegressor(**params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="rmse", callbacks=[early_stopping(250), log_evaluation(0)])
    elif kind == "cb":
        m = CatBoostRegressor(**params)
        m.fit(Xtr, ytr, eval_set=(Xva, yva), early_stopping_rounds=250, use_best_model=True, verbose=0)
    else:
        m = XGBRegressor(**params); m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m
gkf = GroupKFold(n_splits=N_SPLITS); folds = list(gkf.split(X, y, groups=g))
oof = {name: np.zeros(len(y), np.float32) for name, _, _ in VARIANTS}
for name, kind, params in VARIANTS:
    d = MODELS / name; d.mkdir(parents=True, exist_ok=True)
    for k, (tr, va) in enumerate(folds):
        fp = d / f"fold{k}.pkl"
        if fp.exists(): m = joblib.load(fp); tag = "load"
        else:
            m = fit_one(kind, params, X[tr], y[tr], X[va], y[va]); joblib.dump(m, fp); tag = "fit"
        oof[name][va] = m.predict(X[va])
        print(f"{name} fold{k} [{tag}]: rmse={root_mean_squared_error(y[va], oof[name][va]):.4f}", flush=True)
        del m; gc.collect()
    print(f"== {name} OOF rmse={root_mean_squared_error(y, oof[name]):.4f}")

Training until validation scores don't improve for 250 rounds


In [ ]:
# ===== アンサンブル OOF + apply_pp(submit準拠: pf_res=pf_ancc-lk) → overall RMSE =====
names = [v[0] for v in VARIANTS]
oof_ens = np.mean([oof[n] for n in names], axis=0)
w_pf = float(pp_params.get("w_pf", 0.09)); tau = pp_params.get("tau", 85); alpha = float(pp_params.get("alpha", 1.0))
pf_res = pf_abs - lk
def apply_pp_res(res):
    d = res * (1 - w_pf) + pf_res * w_pf
    if tau: d = d * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(tau)))
    return d * alpha
raw = root_mean_squared_error(y, oof_ens); ppr = root_mean_squared_error(y, apply_pp_res(oof_ens))
print("=== OOF overall RMSE(全行プール=Kaggle指標) ===")
print(f"  raw(GBMアンサンブル) : {raw:.4f}")
print(f"  + apply_pp           : {ppr:.4f}")
print("  ※ sg は per-well=submitで適用。CV↔LB は ±0.2 床込みで。")

=== OOF overall RMSE(全行プール=Kaggle指標) ===
  raw(GBMアンサンブル) : 7.0490
  + apply_pp           : 7.0481
  ※ sg は per-well=submitで適用。CV↔LB は ±0.2 床込みで。


In [ ]:
# ===== v29 NN(combo) との残差ブレンド + 公平基準(旧v28 GBM×NN を同一行で再計算) =====
V29_ART  = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v29\artifacts_v29")
V29B_ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v29\artifacts_v29b")
_step = max(1, int(round(1.0 / ROW_FRAC))) if ROW_FRAC < 0.6 else 1
nnz = np.load(V29B_ART / "nn_oof_combo.npz")
nn_ens = np.mean([nnz[k] for k in nnz.files], axis=0)[::_step]
gbm_old = np.load(V29_ART / "gbm_oof.npy")[::_step]
assert len(nn_ens) == len(y) == len(gbm_old), (len(nn_ens), len(y), len(gbm_old))
def overall_pp(res): return root_mean_squared_error(y, apply_pp_res(res))
ws2 = np.arange(0.0, 1.001, 0.02)
scs0 = [overall_pp((1 - w2) * gbm_old + w2 * nn_ens) for w2 in ws2]
b0 = int(np.argmin(scs0))
scs = [overall_pp((1 - w2) * oof_ens + w2 * nn_ens) for w2 in ws2]
bi = int(np.argmin(scs))
print(f"公平基準(旧v28GBM×NN, 同一行): 旧GBM単体 {overall_pp(gbm_old):.4f} / blend w_nn={ws2[b0]:.2f} -> {scs0[b0]:.4f}")
print(f"GBM(v35)単体+pp             : {overall_pp(oof_ens):.4f}")
print(f"blend GBM(v35)×NN           : w_nn={ws2[bi]:.2f} -> {scs[bi]:.4f}  (Δ vs 基準 {scs[bi] - scs0[b0]:+.4f})")
print("※ 行はROW_FRAC間引き後の同一行同士。全行の絶対値(6.8711等)とは僅かにずれるがΔは公平。")
np.save(ART / "gbm_v35_oof.npy", oof_ens)
print("saved:", ART / "gbm_v35_oof.npy")
